# Step 4: XGBoost Dual-Objective Matcher
Trains two XGBoost models:
- **Regressor**: predicts detour distance (km)
- **Classifier**: predicts acceptance probability

Includes grid search hyperparameter tuning, 5-fold cross-validation, and saves feature importance + ROC curve plots.

**To run:** Use `Kernel → Restart & Run All` to execute all cells in order.

In [1]:
# Run from project root
import os
if os.path.basename(os.getcwd()) in ['models', 'simulation', 'data']:
    os.chdir('../..')
elif os.path.basename(os.getcwd()) == 'src':
    os.chdir('..')
print('Working directory:', os.getcwd())

# Enable inline plots for notebook display
%matplotlib inline

Working directory: /Users/krunal/Desktop/final-project-team-tle


In [2]:
import os
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, accuracy_score,
    classification_report, f1_score, roc_auc_score, roc_curve
)

class MultiObjectiveMatcher:
    """
    Dual-objective XGBoost model for P2P ride matching.
    Trains a regressor (detour prediction) and classifier (accept/reject)
    on the same features, then combines their outputs into a match score.
    """
    def __init__(self):
        # Sub-Model A: Predicts physical distance deviation
        self.regressor = xgb.XGBRegressor(
            objective='reg:squarederror', 
            n_estimators=100, 
            learning_rate=0.1, 
            max_depth=5
        )
        
        # Sub-Model B: Predicts boolean acceptance logic
        self.classifier = xgb.XGBClassifier(
            objective='binary:logistic', 
            n_estimators=100, 
            learning_rate=0.1, 
            max_depth=5,
            eval_metric='logloss'
        )
        
        # The exact parameter columns the tree utilizes to compute splits
        self.features = ['driver_orig_dist', 'overlap_ratio', 'time_diff_mins', 'driver_approx_fare', 'rider_approx_fare']
        
    def prepare_data(self, df):
        X = df[self.features]
        y_reg = df['detour_distance_km'] # Target 1 (Continuous)
        y_clf = df['label'] # Target 2 (Categorical/Boolean)
        return X, y_reg, y_clf

    def train(self, df):
        """
        Executes dual-layer algorithm training with a driver-level train/test split.
        Both models evaluate on the exact same test set for consistent scoring.
        Includes 5-fold cross-validation and hyperparameter tuning via grid search.
        """
        X, y_reg, y_clf = self.prepare_data(df)
        print(f"Training on {len(X)} samples with {len(self.features)} features.")

        # Driver-level train/test split to prevent entity leakage.
        # Splitting by pair (random) allows the same driver to appear in both
        # train and test, inflating test accuracy. Splitting by driver_id ensures
        # the model never sees a test driver during training.
        unique_drivers = df['driver_id'].unique()
        train_drivers, test_drivers = train_test_split(
            unique_drivers, test_size=0.2, random_state=42
        )
        train_mask = df['driver_id'].isin(train_drivers)
        train_df = df[train_mask].reset_index(drop=True)
        test_df  = df[~train_mask].reset_index(drop=True)

        X_train = train_df[self.features]
        X_test  = test_df[self.features]
        yr_train = train_df['detour_distance_km']
        yr_test  = test_df['detour_distance_km']
        yc_train = train_df['label']
        yc_test  = test_df['label']

        print(f"Driver-level split: {len(train_df)} train pairs, {len(test_df)} test pairs.")
        print(f"No driver_id overlap between train and test: "
              f"{len(set(train_df['driver_id']) & set(test_df['driver_id'])) == 0}")

        # Hyperparameter tuning via grid search on the classifier 
        print("\nRunning grid search for classifier hyperparameters...")
        param_grid = {
            'n_estimators': [50, 100, 200],
            'max_depth': [3, 5, 7],
            'learning_rate': [0.05, 0.1, 0.2]
        }
        grid_search = GridSearchCV(
            xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss'),
            param_grid,
            cv=5,
            scoring='accuracy',
            n_jobs=-1
        )
        grid_search.fit(X_train, yc_train)
        best_params = grid_search.best_params_
        print(f"Best classifier params: {best_params}")

        # Rebuild both models with tuned params
        self.classifier = xgb.XGBClassifier(
            objective='binary:logistic',
            eval_metric='logloss',
            **best_params
        )
        self.regressor = xgb.XGBRegressor(
            objective='reg:squarederror',
            n_estimators=best_params['n_estimators'],
            max_depth=best_params['max_depth'],
            learning_rate=best_params['learning_rate']
        )

        # 5-fold cross-validation on training set 
        print("\nRunning 5-fold cross-validation...")
        cv_clf_scores = cross_val_score(self.classifier, X_train, yc_train, cv=5, scoring='accuracy')
        cv_reg_scores = cross_val_score(self.regressor, X_train, yr_train, cv=5, scoring='neg_mean_absolute_error')

        print(f"Classifier CV Accuracy: {cv_clf_scores.mean():.4f} (+/- {cv_clf_scores.std():.4f})")
        print(f"  Fold scores: {np.round(cv_clf_scores, 4)}")
        print(f"Regressor CV MAE: {-cv_reg_scores.mean():.4f} (+/- {cv_reg_scores.std():.4f})")
        print(f"  Fold scores: {np.round(-cv_reg_scores, 4)}")

        # Final training on full train set with best params 
        print("\nTraining final models with best hyperparameters...")
        self.regressor.fit(X_train, yr_train)
        pred_detour = self.regressor.predict(X_test)
        mae = mean_absolute_error(yr_test, pred_detour)
        print(f"Regression MAE (Detour km): {mae:.2f}")

        self.classifier.fit(X_train, yc_train)
        pred_match = self.classifier.predict(X_test)
        pred_proba = self.classifier.predict_proba(X_test)[:, 1]
        acc = accuracy_score(yc_test, pred_match)
        print(f"Classification Accuracy: {acc:.2f}")
        print("Classification Report:")
        print(classification_report(yc_test, pred_match, zero_division=0))

        # Additional metrics
        rmse = np.sqrt(mean_squared_error(yr_test, pred_detour))
        f1 = f1_score(yc_test, pred_match, zero_division=0)
        roc_auc = roc_auc_score(yc_test, pred_proba)
        print(f"\nRegression RMSE (Detour km): {rmse:.4f}")
        print(f"Classification F1: {f1:.4f}")
        print(f"Classification ROC-AUC: {roc_auc:.4f}")

        # Save plots
        os.makedirs("figures", exist_ok=True)

        # Feature importance
        fig, ax = plt.subplots(figsize=(8, 5))
        xgb.plot_importance(self.classifier, ax=ax, importance_type='gain')
        ax.set_title("Feature Importance (Classifier -Gain)")
        fig.tight_layout()
        fig.savefig("figures/feature_importance.png", dpi=150)
        plt.show()
        plt.close(fig)
        print("Saved figures/feature_importance.png")

        # ROC curve
        fpr, tpr, _ = roc_curve(yc_test, pred_proba)
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.plot(fpr, tpr, label=f"XGBoost (AUC = {roc_auc:.3f})")
        ax.plot([0, 1], [0, 1], 'k--', label="Random")
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.set_title("ROC Curve -Match Acceptance Classifier")
        ax.legend()
        fig.tight_layout()
        fig.savefig("figures/roc_curve.png", dpi=150)
        plt.show()
        plt.close(fig)
        print("Saved figures/roc_curve.png")

        return X_test, yr_test, yc_test, pred_detour, pred_match

    def compute_match_score(self, feature_row):
        """Combine detour prediction and acceptance probability into a single match score."""
        # Formulate pandas DataFrame single entry safely
        df_feats = pd.DataFrame([feature_row], columns=self.features)
        
        predicted_detour = self.regressor.predict(df_feats)[0]
        match_probability = self.classifier.predict_proba(df_feats)[0][1] # Pull exact float percentage
        
        # Strict algorithmic penalty system:
        # Subtracting heavily weighted detour values directly from probabilistic intent bounds.
        score = match_probability - (predicted_detour * 0.1) 
        return score

In [3]:
# Run the XGBoost dual-objective matcher
print("Loading simulated candidate pairs...")
df = pd.read_parquet("data/simulated/candidate_pairs.parquet")
matcher = MultiObjectiveMatcher()
X_test, yr_test, yc_test, pred_detour, pred_match = matcher.train(df)

Loading simulated candidate pairs...
Training on 750 samples with 5 features.
Driver-level split: 592 train pairs, 158 test pairs.
No driver_id overlap between train and test: True

Running grid search for classifier hyperparameters...


Best classifier params: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 50}

Running 5-fold cross-validation...


Classifier CV Accuracy: 0.9412 (+/- 0.1135)
  Fold scores: [0.7143 0.9916 1.     1.     1.    ]
Regressor CV MAE: 3.0651 (+/- 1.6615)
  Fold scores: [1.4    1.0027 3.1205 4.7774 5.0252]

Training final models with best hyperparameters...
Regression MAE (Detour km): 2.75
Classification Accuracy: 0.91
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.82      0.90        85
           1       0.83      1.00      0.91        73

    accuracy                           0.91       158
   macro avg       0.91      0.91      0.91       158
weighted avg       0.92      0.91      0.90       158


Regression RMSE (Detour km): 5.2083
Classification F1: 0.9068
Classification ROC-AUC: 0.9661
Saved figures/feature_importance.png
Saved figures/roc_curve.png


/var/folders/xc/wspxd6_n7k90gnw05920r_yw0000gn/T/ipykernel_80629/7237321.py:152: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/xc/wspxd6_n7k90gnw05920r_yw0000gn/T/ipykernel_80629/7237321.py:167: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
